In [3]:
# --- Prime Sequence Decipher Game (PSD) for Google Colab ---
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
import math

# ----- Game State -----
secret_number_str = None
secret_digits = []
secret_sum = 0
low, high = 1, 9  # Default Easy
attempts = 0
score_history = []  # Stores (range_high, attempts, avg_prime_accuracy)
prime_accuracy_sum = 0 # To track total accuracy across all attempts

# ----- Utility Functions -----

def is_prime(n):
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

def get_prime_composite_status(digits):
    """Returns a list of booleans: True if Prime, False if Composite/1"""
    return [is_prime(d) for d in digits]

# ----- UI Elements -----

# Difficulty buttons
easy_btn = widgets.Button(description="Easy (1–9)", button_style="success")
medium_btn = widgets.Button(description="Medium (1–15)", button_style="info")
hard_btn = widgets.Button(description="Hard (1–25)", button_style="danger")
difficulty_box = widgets.HBox([easy_btn, medium_btn, hard_btn])

guess_box = widgets.Text(description="Guess (3-digits):", layout=widgets.Layout(width="250px"), placeholder="E.g., 419")
submit_btn = widgets.Button(description="Decipher", button_style="info")
restart_btn = widgets.Button(description="Restart", button_style="warning")

progress_bar = widgets.FloatProgress(value=0, min=0, max=1.0, description="Sum Closeness:", bar_style='')
score_out = widgets.Output()
message_out = widgets.Output()


# ----- Game Logic -----
def set_difficulty(level):
    global low, high
    if level == "easy":
        high = 9
    elif level == "medium":
        high = 15
    elif level == "hard":
        high = 25
    low = 1
    start_game(level)


def start_game(difficulty_name="medium"):
    """Start/reset the game with chosen difficulty."""
    global secret_digits, secret_number_str, secret_sum, attempts, prime_accuracy_sum
    
    # Generate 3 random digits based on range
    secret_digits = [random.randint(low, high) for _ in range(3)]
    secret_number_str = "".join(map(str, secret_digits))
    secret_sum = sum(secret_digits)
    
    attempts = 0
    prime_accuracy_sum = 0
    
    progress_bar.value = 0
    progress_bar.bar_style = ""

    guess_box.value = ""
    message_out.clear_output()

    with message_out:
        print(f"**Game Started: Prime Sequence Decipher**")
        print(f"Difficulty: {difficulty_name.capitalize()}. Decipher a 3-digit sequence where each digit is between {low} and {high}.")
        print(f"Enter your 3-digit guess (e.g., 105 for digits 10, 5)!")


def submit_guess(_):
    """Handle guess submission and provide complex feedback."""
    global attempts, prime_accuracy_sum

    guess_str = guess_box.value.strip()
    
    # --- Input Validation ---
    message_out.clear_output() # Clear for a clean new turn display

    if not guess_str.isdigit() or len(guess_str) < 3:
        with message_out:
            print("❌ Error: Enter a valid sequence of digits (e.g., 419).")
        return

    # Check if individual digits are within the set range (complex validation)
    try:
        guess_digits = [int(digit) for digit in guess_str] # Simple parsing for input like '123' -> [1, 2, 3]

        if not (all(low <= d <= high for d in guess_digits) or 
                (len(guess_str) > 3 and all(low <= int(d) <= high for d in [guess_str[:1], guess_str[1:2], guess_str[2:]]))):
            
            # Simple three digit case, re-parse as single digits for the game logic
            if len(guess_str) == 3:
                 guess_digits = [int(guess_str[i]) for i in range(3)]

            # More complex parsing logic is needed if max is > 9 (e.g. 101525)
            # Simplification: Only allow 3 digits, and rely on the number of digits in the secret_digits list
            pass
            
    except ValueError:
        with message_out:
            print(f"❌ Error: Could not interpret digits.")
        return

    # To handle the display: we rely on the number of secret digits (3)
    if len(guess_str) != len(secret_number_str):
        with message_out:
            print(f"❌ Error: Your guess length must match the hidden number's digit count ({len(secret_number_str)}).")
        return

    # Dynamic Digit Parsing: Break the guess string into chunks matching secret_digits length
    # This is complex when high > 9 (e.g., guessing 101112 for secret 10, 11, 12)
    # FOR SIMPLICITY, we stick to N=3 digits from 1-9 for robust validation.
    # To support high > 9, the input must be formatted, e.g., "10-11-12", but we'll use simple string input for UI.
    
    if high <= 9: # Simple case: each character is a digit
        guess_digits = [int(guess_str[i]) for i in range(3)]
    else:
         # Simplified for complex ranges: require user to input a fixed-width sequence or rely on exact match
         # Since this is a demonstration, we will skip the complex parsing for digits > 9 and stick to the logic below:
         # If high > 9, we assume the user must guess the *exact concatenated number*.
         if guess_str != secret_number_str:
            with message_out:
                print(f"❌ Error: Invalid digit length for this difficulty. Guess must be the full concatenated number.")
            return # This simplifies the game logic immensely, but preserves the original concept.
         else:
            guess_digits = secret_digits
            
    
    # Final check on range after simplified parsing
    if not all(low <= d <= high for d in guess_digits):
        with message_out:
            print(f"❌ Error: Each digit must be between {low} and {high}.")
        return


    attempts += 1
    
    # --- Hint Calculations ---
    
    # Sum Closeness Hint
    guess_sum = sum(guess_digits)
    sum_diff = abs(secret_sum - guess_sum)
    max_sum_diff = (high - low) * 3 # Maximum possible difference in sum
    
    sum_closeness_ratio = 1 - sum_diff / max_sum_diff
    progress_bar.value = max(0, min(sum_closeness_ratio, 1))

    if sum_diff == 0:
        sum_hint = "🎯 Perfect Sum Match!"
        progress_bar.bar_style = "success"
    elif sum_diff <= max_sum_diff * 0.15: # 15% difference
        sum_hint = f"Sum is Very Hot! ({secret_sum})"
        progress_bar.bar_style = "info"
    elif sum_diff <= max_sum_diff * 0.40: # 40% difference
        sum_hint = f"Sum is Warm. ({secret_sum})"
        progress_bar.bar_style = "warning"
    else:
        sum_hint = f"Sum is Cold. ({secret_sum})"
        progress_bar.bar_style = "danger"

    # Prime/Composite Status Match (Property B)
    secret_status = get_prime_composite_status(secret_digits)
    guess_status = get_prime_composite_status(guess_digits)
    
    prime_matches = sum(s == g for s, g in zip(secret_status, guess_status))
    
    # Track performance for score calculation
    prime_accuracy_sum += prime_matches / 3

    # Digit Order Match
    order_matches = sum(1 for gd, sd in zip(guess_digits, secret_digits) if gd == sd)

    # --- Print Feedback ---
    
    if guess_str == secret_number_str:
        # Final Win Output
        print(f"🎉 DECIPHERED! The secret sequence was {secret_number_str}.")
        print(f"Total Attempts: {attempts}")
        
        score_history.append({
            "range": high, 
            "attempts": attempts, 
            "avg_prime_accuracy": prime_accuracy_sum / attempts
        })
        update_score_history()
        return

    # Print Hints for an incorrect guess
    print(f"Guess: {guess_str}")
    print(f"1. Sum Closeness: {sum_hint}")
    print(f"2. Prime Status Match: {prime_matches} of 3 digits match the secret's Prime/Composite property.")
    print(f"3. Digit Match: {order_matches} digit(s) are correct and in the right position.")
    print(f"Attempts: {attempts}")

        
def update_score_history():
    """Refresh the displayed score history."""
    score_out.clear_output()
    with score_out:
        print("🏆 Score History:")
        if not score_history:
            print("No completed games yet.")
            return
            
        for i, entry in enumerate(score_history, start=1):
            prime_acc = entry['avg_prime_accuracy'] * 100
            
            if prime_acc >= 75:
                rating = "Strategist"
            elif prime_acc >= 50:
                rating = "Analyst"
            else:
                rating = "Trial-and-Error"
                
            print(f"{i}. Range 1-{entry['range']} -> Attempts: {entry['attempts']} | Prime Accuracy: {prime_acc:.1f}% ({rating})")


def restart_game(_):
    # This restarts the game with the *last* selected difficulty range
    start_game() 


# ----- Event Wiring -----
easy_btn.on_click(lambda _: set_difficulty("easy"))
medium_btn.on_click(lambda _: set_difficulty("medium"))
hard_btn.on_click(lambda _: set_difficulty("hard"))
submit_btn.on_click(submit_guess)
restart_btn.on_click(restart_game)


# ----- Display UI -----
ui = widgets.VBox([
    widgets.HTML("<h2>👾 Prime Sequence Decipher (PSD)</h2>"),
    widgets.HTML("<b>Choose Digit Range:</b>"),
    difficulty_box,
    guess_box,
    submit_btn,
    restart_btn,
    progress_bar,
    message_out,
    widgets.HTML("<hr>"),
    score_out
])

display(ui)

# Initial Start
start_game("easy")

In [4]:
# HexaGrid Deduction — Google Colab / IPython widget implementation
# Run this whole cell in a Colab notebook.

import random
import time
import math
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from collections import deque

# ------------------------
# Game configuration
# ------------------------
GRID_SIZE = 4
MIN_VAL = 1
MAX_VAL = 9
STARTING_ENERGY = 30

# ------------------------
# Game state
# ------------------------
hidden_grid = np.random.randint(MIN_VAL, MAX_VAL+1, size=(GRID_SIZE, GRID_SIZE))
# For reproducible testing uncomment:
# random.seed(42); np.random.seed(42); hidden_grid = np.random.randint(MIN_VAL, MAX_VAL+1, size=(GRID_SIZE, GRID_SIZE))

guesses = [[None for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
revealed = [[False for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
energy = STARTING_ENERGY
move_history = deque(maxlen=200)
reveal_count = 0
start_time = None
game_over = False
score_history = []  # store past games if play multiple rounds

# ------------------------
# Helper functions
# ------------------------
def neighbors_coords(r, c):
    coords = []
    for dr in (-1,0,1):
        for dc in (-1,0,1):
            if dr==0 and dc==0: continue
            nr, nc = r+dr, c+dc
            if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
                coords.append((nr,nc))
    return coords

def row_sum(r):
    return int(hidden_grid[r,:].sum())

def col_sum(c):
    return int(hidden_grid[:,c].sum())

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

# ------------------------
# UI elements
# ------------------------
# Grid display: show guesses or revealed value or blank
grid_buttons = [[widgets.Button(description="?", layout=widgets.Layout(width='48px', height='48px'))
                 for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
grid_box = widgets.GridBox([btn for row in grid_buttons for btn in row],
                           layout=widgets.Layout(grid_template_columns=" ".join(["48px"]*GRID_SIZE),
                                                 grid_gap='6px 6px'))

# Controls
coord_r = widgets.BoundedIntText(value=0, min=0, max=GRID_SIZE-1, description='Row')
coord_c = widgets.BoundedIntText(value=0, min=0, max=GRID_SIZE-1, description='Col')
guess_input = widgets.BoundedIntText(value=1, min=MIN_VAL, max=MAX_VAL, description='Guess')
action_sel = widgets.Dropdown(options=[
    ('Probe Cell (cost 1)', 'probe'),
    ('Scan Row (cost 3)','scan_row'),
    ('Scan Col (cost 3)','scan_col'),
    ('Neighborhood Hint (cost 2)','neighbor'),
    ('Reveal Cell (cost 6)','reveal'),
    ('Auto-suggest (cost 4)','autosuggest'),
    ('Submit Grid (end game)','submit')], description='Action')

do_action_btn = widgets.Button(description='Do Action', button_style='primary')
hint_out = widgets.Output(layout=widgets.Layout(height='140px', overflow='auto'))
status_out = widgets.Output()
history_out = widgets.Output(layout=widgets.Layout(height='140px', overflow='auto'))
performance_out = widgets.Output()
plot_out = widgets.Output()

# Live metrics labels
energy_label = widgets.Label(f'Energy: {energy}/{STARTING_ENERGY}')
reveals_label = widgets.Label(f'Reveals used: {reveal_count}')

# ------------------------
# Game logic functions
# ------------------------
def refresh_grid_display():
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            btn = grid_buttons[r][c]
            if revealed[r][c]:
                btn.description = str(hidden_grid[r,c])
                btn.button_style = 'success'
            else:
                val = guesses[r][c]
                btn.description = str(val) if val is not None else '?'
                btn.button_style = ''
            # highlight selected cell
            if r == coord_r.value and c == coord_c.value:
                btn.style.font_weight = 'bold'
            else:
                btn.style.font_weight = 'normal'

def push_history(text):
    global move_history
    t = time.strftime('%H:%M:%S')
    move_history.appendleft(f"[{t}] {text}")
    with history_out:
        clear_output(wait=True)
        for line in list(move_history)[:60]:
            print(line)

def deduct_energy(amount):
    global energy
    if energy < amount:
        with hint_out:
            print(f"Not enough energy for that action (need {amount}, have {energy}).")
        return False
    energy -= amount
    energy_label.value = f'Energy: {energy}/{STARTING_ENERGY}'
    return True

def action_probe_cell(r, c, guess_val=None):
    """Probe: if player gives a guess, return precise abs difference. Else give bucketed hint."""
    if not deduct_energy(1):
        return
    if guess_val is None:
        true = int(hidden_grid[r,c])
        diff = abs(true - ((MIN_VAL+MAX_VAL)//2))  # not used but kept
        # bucket hint based on actual value distance from mid-range 5
        # give closeness bucket relative to center of range or not — we compute real diff from either median guess or last guess
        # But more usefully we notify closeness relative to median 5
        truev = hidden_grid[r,c]
        # Bucket by distance to median 5:
        d = abs(truev - 5)
        if d <= 1:
            hint = "Very Close to middle values"
        elif d <= 2:
            hint = "Close to middle"
        elif d <= 3:
            hint = "Moderate"
        else:
            hint = "Outlier (far from middle)"
        push_history(f"Probe cell ({r},{c}) -> bucket: {hint} (cost 1)")
        with hint_out:
            print(f"Probe result (bucket): {hint}")
    else:
        if not (MIN_VAL <= guess_val <= MAX_VAL):
            with hint_out:
                print(f"Guess must be between {MIN_VAL} and {MAX_VAL}.")
            return
        true = int(hidden_grid[r,c])
        diff = abs(true - guess_val)
        push_history(f"Probe cell ({r},{c}) with guess {guess_val} -> abs diff {diff} (cost 1)")
        with hint_out:
            print(f"Probe result: absolute difference = {diff}")
    refresh_grid_display()

def action_scan_row(r):
    if not deduct_energy(3): return
    s = row_sum(r)
    parity = 'even' if s % 2 == 0 else 'odd'
    push_history(f"Scan row {r} -> sum {s}, parity {parity} (cost 3)")
    with hint_out:
        print(f"Row {r} — sum: {s}, parity: {parity}")

def action_scan_col(c):
    if not deduct_energy(3): return
    s = col_sum(c)
    parity = 'even' if s % 2 == 0 else 'odd'
    push_history(f"Scan col {c} -> sum {s}, parity {parity} (cost 3)")
    with hint_out:
        print(f"Col {c} — sum: {s}, parity: {parity}")

def action_neighbor(r,c):
    if not deduct_energy(2): return
    coords = neighbors_coords(r,c)
    if not coords:
        push_history(f"Neighbor hint ({r},{c}) -> none (cost 2)")
        with hint_out:
            print("No neighbors.")
        return
    avg = round(sum(int(hidden_grid[nr,nc]) for nr,nc in coords) / len(coords))
    push_history(f"Neighbor hint ({r},{c}) -> avg {avg} (cost 2)")
    with hint_out:
        print(f"Neighbors average (rounded): {avg}")

def action_reveal(r,c):
    global reveal_count
    if not deduct_energy(6): return
    if not revealed[r][c]:
        revealed[r][c] = True
        reveal_count += 1
        reveals_label.value = f'Reveals used: {reveal_count}'
        push_history(f"Reveal cell ({r},{c}) -> {int(hidden_grid[r,c])} (cost 6)")
        with hint_out:
            print(f"Revealed cell ({r},{c}) = {int(hidden_grid[r,c])}")
    else:
        with hint_out:
            print("Cell already revealed.")
    refresh_grid_display()

def action_autosuggest(r,c):
    if not deduct_energy(4): return
    # Heuristic: use row sum and known guessed cells to propose best candidate
    # Gather constraints from scans in history if any (not tracked); instead use simple heuristic:
    known_row = [guesses[r][j] if guesses[r][j] is not None else (int(hidden_grid[r,j]) if revealed[r][j] else None) for j in range(GRID_SIZE)]
    known_vals = [v for v in known_row if v is not None]
    if known_vals:
        # suggest median of remaining plausible values based on neighbor averages if exist
        neighs = [int(hidden_grid[nr,nc]) for nr,nc in neighbors_coords(r,c) if revealed[nr][nc]]
        if neighs:
            suggestion = round(sum(neighs)/len(neighs))
        else:
            suggestion = int(round(sum(known_vals)/len(known_vals)))
    else:
        # fallback random heuristic near center
        suggestion = (MIN_VAL + MAX_VAL)//2
    suggestion = clamp(suggestion, MIN_VAL, MAX_VAL)
    push_history(f"Auto-suggest for ({r},{c}) -> {suggestion} (cost 4)")
    with hint_out:
        print(f"Auto-suggest: consider trying {suggestion} for cell ({r},{c}).")
    # does NOT set guess automatically; player must accept
    refresh_grid_display()

def submit_and_score():
    global game_over
    game_over = True
    elapsed = max(0, time.time() - start_time)
    # total absolute error
    total_abs_error = 0
    reveals = reveal_count
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            g = guesses[r][c]
            if g is None:
                # treat empty guess as worst (choose MIN_VAL as default) -> penalize
                g = MIN_VAL
            total_abs_error += abs(int(hidden_grid[r,c]) - int(g))
    max_possible_error = (MAX_VAL - MIN_VAL) * GRID_SIZE * GRID_SIZE / 2  # not perfect; we'll use 8*16
    max_possible_error = 8 * GRID_SIZE * GRID_SIZE
    accuracy_score = 0
    if total_abs_error <= max_possible_error:
        accuracy_score = 100 * (1 - (total_abs_error / max_possible_error))
    else:
        accuracy_score = 0
    energy_bonus = (energy / STARTING_ENERGY) * 20
    reveal_penalty = reveals * 3
    time_penalty = min(10, elapsed / 10)
    final = accuracy_score + energy_bonus - reveal_penalty - time_penalty
    final = clamp(final, 0, 130)

    # Record history
    score_record = {
        'time': time.strftime('%Y-%m-%d %H:%M:%S'),
        'accuracy_score': round(accuracy_score,2),
        'energy_bonus': round(energy_bonus,2),
        'reveal_penalty': reveal_penalty,
        'time_penalty': round(time_penalty,2),
        'final_score': round(final,2),
        'total_abs_error': total_abs_error
    }
    score_history.append(score_record)

    # Output results, heatmaps
    with performance_out:
        clear_output(wait=True)
        print("=== Final Results ===")
        print(f"Elapsed time: {int(elapsed)} s")
        print(f"Total absolute error: {total_abs_error}")
        print(f"Accuracy score component: {accuracy_score:.2f} / 100")
        print(f"Energy bonus: {energy_bonus:.2f} / 20")
        print(f"Reveal penalty: -{reveal_penalty}")
        print(f"Time penalty: -{time_penalty:.2f}")
        print(f"FINAL SCORE: {final:.2f} (max 130)")
        print("\nScore breakdown (most recent at top):")
        for rec in reversed(score_history[-6:]):
            print(rec)
        # plots: true grid, guessed grid, error matrix
        fig, axes = plt.subplots(1,3, figsize=(12,4))
        true_mat = hidden_grid
        guess_mat = np.array([[g if g is not None else np.nan for g in row] for row in guesses])
        err_mat = np.abs(true_mat - np.nan_to_num(guess_mat, MIN_VAL))
        im0 = axes[0].imshow(true_mat, vmin=MIN_VAL, vmax=MAX_VAL)
        axes[0].set_title('True grid')
        for (i,j), val in np.ndenumerate(true_mat):
            axes[0].text(j,i,str(val), ha='center', va='center', color='white')
        im1 = axes[1].imshow(np.nan_to_num(guess_mat, MIN_VAL), vmin=MIN_VAL, vmax=MAX_VAL)
        axes[1].set_title('Your guesses (empty->min)')
        for (i,j), val in np.ndenumerate(guess_mat):
            txt = str(int(val)) if not np.isnan(val) else '-'
            axes[1].text(j,i,txt, ha='center', va='center', color='white')
        im2 = axes[2].imshow(err_mat, vmin=0, vmax=MAX_VAL-MIN_VAL)
        axes[2].set_title('Absolute error per cell')
        for (i,j), val in np.ndenumerate(err_mat):
            axes[2].text(j,i,str(int(val)), ha='center', va='center', color='white')
        fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
        plt.show()

def on_grid_button_clicked(r,c):
    # When a grid cell bright clicked, set controls to that cell
    coord_r.value = r
    coord_c.value = c
    refresh_grid_display()

# attach callbacks to grid buttons
for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        def make_cb(rr,cc):
            def cb(btn):
                on_grid_button_clicked(rr,cc)
            return cb
        grid_buttons[r][c].on_click(make_cb(r,c))

# ------------------------
# Wire action button
# ------------------------
def on_do_action(_):
    global start_time, reveal_count, game_over
    if game_over:
        with hint_out:
            print("Game already finished. Restart to play again.")
        return
    if start_time is None:
        start_time = time.time()
    act = action_sel.value
    r = int(coord_r.value)
    c = int(coord_c.value)
    if act == 'probe':
        # allow probing with optional guess
        g = guess_input.value
        # player can choose to probe with or without purposeful guess: if guess_input equals None? we always pass a guess here
        action_probe_cell(r,c, guess_val=g)
    elif act == 'scan_row':
        action_scan_row(r)
    elif act == 'scan_col':
        action_scan_col(c)
    elif act == 'neighbor':
        action_neighbor(r,c)
    elif act == 'reveal':
        action_reveal(r,c)
    elif act == 'autosuggest':
        action_autosuggest(r,c)
    elif act == 'submit':
        submit_and_score()
    refresh_grid_display()
    push_history(f"Energy left: {energy}")

do_action_btn.on_click(on_do_action)

# ------------------------
# Guess input handling (no energy)
# ------------------------
def on_guess_change(change):
    # change the guesses matrix directly to allow player to type guesses
    r = int(coord_r.value)
    c = int(coord_c.value)
    val = change['new']
    if val is None:
        return
    val = int(clamp(int(val), MIN_VAL, MAX_VAL))
    guesses[r][c] = val
    push_history(f"Set guess for ({r},{c}) = {val}")
    refresh_grid_display()

guess_input.observe(on_guess_change, names='value')

# ------------------------
# Restart / New game button
# ------------------------
restart_btn = widgets.Button(description='Restart Game (new grid)', button_style='warning')
def on_restart(_):
    global hidden_grid, guesses, revealed, energy, move_history, reveal_count, start_time, game_over
    hidden_grid = np.random.randint(MIN_VAL, MAX_VAL+1, size=(GRID_SIZE, GRID_SIZE))
    guesses = [[None for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
    revealed = [[False for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
    energy = STARTING_ENERGY
    move_history = deque(maxlen=200)
    reveal_count = 0
    start_time = None
    game_over = False
    energy_label.value = f'Energy: {energy}/{STARTING_ENERGY}'
    reveals_label.value = f'Reveals used: {reveal_count}'
    with hint_out:
        clear_output()
        print("New grid generated. Good luck!")
    refresh_grid_display()
    with history_out:
        clear_output()
on_restart_btn = restart_btn.on_click(on_restart)

# ------------------------
# Layout assemble
# ------------------------
control_col = widgets.VBox([
    widgets.HTML("<b>Choose cell coordinates (click grid to auto-fill)</b>"),
    widgets.HBox([coord_r, coord_c]),
    guess_input,
    widgets.HBox([action_sel, do_action_btn]),
    widgets.HBox([energy_label, reveals_label]),
    restart_btn
])

left_col = widgets.VBox([grid_box, widgets.HTML("<i>Click a cell to select it; then pick an action.</i>")])
right_col = widgets.VBox([control_col, widgets.HTML("<b>Hints & Messages</b>"), hint_out, widgets.HTML("<b>Move History</b>"), history_out])

top = widgets.HBox([left_col, right_col])
bottom = widgets.VBox([widgets.HTML("<h4>Performance & Results</h4>"), performance_out])

main_ui = widgets.VBox([widgets.HTML("<h3>HexaGrid Deduction</h3><small>Deduce a 4×4 grid of integers 1–9 with probes and limited energy.</small>"),
                        top, bottom])

# Initialize displays
with hint_out:
    print("Welcome to HexaGrid Deduction!")
    print("Select a cell then choose an action. Use 'Submit Grid' once you are ready.")
refresh_grid_display()
display(main_ui)


In [8]:
# --- FIXED Quantum Echo Pattern Hunter ---
# Corrected for Colab/Jupyter - Handles mixed types & edge cases
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# ----- Game State -----
game_state = {
    'pattern': None,
    'dimension': 3,
    'attempts': 0,
    'max_attempts': 12,
    'history': [],
    'score': 0,
    'level': 1
}

# Fixed patterns - consistent typing
def generate_pattern(level):
    if level == 1:
        return [random.randint(1,9) for _ in range(3)]
    elif level == 2:
        return [random.randint(1,15) for _ in range(4)] + [random.choice(['A','B','C'])]
    else:
        return [random.randint(1,25) for _ in range(5)] + [random.choice(['X','Y','Z','W'])]

# ----- UI Elements -----
level_btns = widgets.ToggleButtons(
    options=[('Echo Hunt (3D)', 1), ('Quantum Echo (4D)', 2), ('Dimensional Rift (5D)', 3)],
    description='Level:', button_style='info'
)

# Fixed dropdowns - proper options per level
def create_guess_inputs(dimension):
    return [widgets.Dropdown(options=[''] + list(range(1,26)) + ['A','B','C','X','Y','Z','W'], 
                            layout=widgets.Layout(width='70px')) for _ in range(dimension)]

guess_inputs = create_guess_inputs(3)
guess_box = widgets.HBox(guess_inputs)

submit_btn = widgets.Button(description="Echo Scan", button_style="success")
hint_btn = widgets.Button(description="Echo Pulse", button_style="warning")
restart_btn = widgets.Button(description="Reset Rift", button_style="danger")

progress_bar = widgets.FloatProgress(value=0, min=0, max=1, description="Pattern Lock:", bar_style='')
status_out = widgets.Output()
history_out = widgets.Output()
score_out = widgets.Output()
echo_visual = widgets.Output()

input_box = widgets.VBox([guess_box, widgets.HBox([submit_btn, hint_btn])])

# ----- FIXED Game Mechanics -----
def safe_numeric_diff(a, b):
    """Handle str/int comparison safely"""
    if isinstance(a, str) or isinstance(b, str):
        return 999  # Large difference for non-numeric
    return abs(int(a) - int(b))

def calculate_echo_distance(guess, target):
    """Fixed echo feedback - handles mixed types"""
    score = 0
    hints = []
    
    # Ensure same length
    min_len = min(len(guess), len(target))
    
    # Position + Value echo (50% weight)
    for i in range(min_len):
        g, t = guess[i], target[i]
        if g == t: 
            score += 2
            hints.append(f"Pos{i+1}: Perfect")
        elif g and str(g)[-1] == str(t)[-1]:  # Echo resonance (last character)
            score += 1
            hints.append(f"Pos{i+1}: Echo")
    
    # Pattern momentum echo (30% weight) - FIXED numeric only
    momentum = 0
    numeric_pairs = 0
    for i in range(min_len):
        if isinstance(guess[i], (int, str)) and str(guess[i]).isdigit() and str(target[i]).isdigit():
            momentum += safe_numeric_diff(guess[i], target[i]) ** 2
            numeric_pairs += 1
    
    if numeric_pairs > 0 and momentum / numeric_pairs < 10:
        score += 1
        hints.append("Momentum")
    
    # Dimensional shift echo (20% weight)
    if len(set(str(g) for g in guess[:3] if g)) == len(set(str(t) for t in target[:3])):
        score += 1
        hints.append("Shift")
    
    return score, hints

def update_visualization():
    """Dynamic echo visualization"""
    with echo_visual:
        clear_output()
        if game_state['history']:
            viz = f"🌊 Echo Field: {game_state['attempts']}/{game_state['max_attempts']}\n"
            viz += "═" * 50 + "\n"
            for i, h in enumerate(game_state['history'][-3:], 1):
                bar = "█" * min(int(h['echo_score']*5), 10) + "░" * max(0, 10-int(h['echo_score']*5))
                hints_str = ", ".join(h.get('hints', []))[:20]
                viz += f"{i}. {bar} [{h['echo_score']:.1f}] {hints_str}\n"
            print(viz)

def start_game(change=None):
    """Initialize echo pattern hunt - FIXED"""
    level = level_btns.value if hasattr(level_btns, 'value') else 1
    dimension = min(3 + (level-1), 5)
    
    game_state.update({
        'pattern': generate_pattern(level),
        'attempts': 0, 
        'history': [], 
        'score': game_state['score'],  # Preserve total score
        'dimension': dimension,
        'max_attempts': 8 + (level-1)*2,
        'level': level
    })
    
    # Recreate inputs for current dimension
    global guess_inputs, guess_box
    guess_inputs = create_guess_inputs(dimension)
    guess_box.children = [widgets.HBox(guess_inputs)]
    
    for inp in guess_inputs: 
        inp.value = ''
    
    progress_bar.value = 0
    progress_bar.bar_style = ''
    [out.clear_output() for out in [status_out, history_out, score_out, echo_visual]]
    
    with status_out:
        print(f"🔮 RIFT OPENED: Level {level}")
        print(f"📏 Hunt {dimension}D Echo Pattern: {len(game_state['pattern'])} positions")
        print(f"⏱️  {game_state['max_attempts']} Echo Scans Max")
        print("\n🎯 Predict the multidimensional resonance pattern!")

def submit_echo_scan(b):
    """Process pattern prediction - FIXED error handling"""
    guess = [w.value for w in guess_inputs]
    
    # Validate complete guess
    if '' in guess or None in guess:
        with status_out: 
            print("❌ Complete all echo positions!")
        return
    
    if game_state['attempts'] >= game_state['max_attempts']:
        with status_out:
            print(f"💥 RIFT CLOSED! Pattern was: {game_state['pattern']}")
        return
    
    game_state['attempts'] += 1
    echo_score_raw, hints = calculate_echo_distance(guess, game_state['pattern'])
    echo_score = echo_score_raw / max(len(game_state['pattern']), 1)
    
    game_state['history'].append({
        'guess': guess, 
        'echo_score': echo_score,
        'hints': hints,
        'raw_score': echo_score_raw
    })
    
    # Update progress
    total_echo = sum(h['echo_score'] for h in game_state['history'])
    progress_bar.value = min(1, total_echo / (game_state['max_attempts'] * 0.8))
    progress_bar.bar_style = 'success' if progress_bar.value > 0.7 else 'warning' if progress_bar.value > 0.4 else 'danger'
    
    # Victory condition
    max_possible = len(game_state['pattern']) * 2
    if echo_score_raw >= max_possible or total_echo > game_state['max_attempts'] * 1.2:
        bonus = max(0, (game_state['max_attempts'] - game_state['attempts'] + 1) * 10)
        game_state['score'] += int(bonus + echo_score_raw * 5)
        
        with status_out:
            print(f"\n🎉 RIFT SEALED! Pattern: {game_state['pattern']}")
            print(f"🏆 Perfect Echo: {echo_score_raw}/{max_possible}")
            print(f"⭐ Session Bonus: +{int(bonus)} | Total: {game_state['score']}")
        update_leaderboard()
        return
    
    update_visualization()
    update_leaderboard()
    
    with status_out:
        print(f"\n📡 Scan {game_state['attempts']}: Echo {echo_score_raw:.1f}/{max_possible}")
        if hints: 
            print("💫 Resonances:", ", ".join(hints))
        remaining = game_state['max_attempts'] - game_state['attempts']
        print(f"⏳ {remaining} scans left...")

def echo_pulse(b):
    """Directional hint: higher / lower / very close, without revealing the answer"""
    if not game_state['history']:
        with status_out:
            print("⚡ Pulse first after a scan!")
        return

    last_guess = game_state['history'][-1]['guess']
    target = game_state['pattern']
    min_len = min(len(last_guess), len(target))

    best_pos = None
    best_diff = float('inf')

    # Find closest NON-perfect numeric position
    for i in range(min_len):
        g, t = last_guess[i], target[i]

        # Only for numeric positions on both sides
        if isinstance(g, (int, str)) and isinstance(t, (int, str)) \
           and str(g).isdigit() and str(t).isdigit():

            d = abs(int(g) - int(t))
            if d != 0 and d < best_diff:
                best_diff = d
                best_pos = i

    with status_out:
        if best_pos is None:
            print("⚡ PULSE: No numeric position is close enough yet. Try changing more values.")
            return

        g = int(last_guess[best_pos])
        t = int(target[best_pos])

        # Decide direction text
        if g < t:
            direction = "higher"
        else:
            direction = "lower"

        # Decide how strong the hint is
        if best_diff <= 2:
            closeness = "just a bit"
        elif best_diff <= 5:
            closeness = "somewhat"
        else:
            closeness = "much"

        print(f"⚡ PULSE: Position {best_pos+1} is {closeness} off, try {direction}.")

def update_leaderboard():
    """Enhanced score tracking"""
    with history_out:
        clear_output()
        if game_state['history']:
            print("📊 Recent Echoes:")
            for i, entry in enumerate(game_state['history'][-5:], 1):
                g_str = ' '.join(map(str, entry['guess']))
                print(f"{i}. {g_str} → {entry['raw_score']:.1f} ({', '.join(entry.get('hints',[]))})")
    
    with score_out:
        clear_output()
        print(f"🌟 Career Score: {game_state['score']}")
        print(f"📈 Level {game_state['level']} | Best: {max([h.get('raw_score',0) for h in game_state['history']]):.1f}")

# ----- Event Handlers -----
def on_level_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        start_game()

level_btns.observe(on_level_change, 'value')
submit_btn.on_click(submit_echo_scan)
hint_btn.on_click(echo_pulse)
restart_btn.on_click(start_game)

# ----- Rich UI Layout -----
game_ui = widgets.VBox([
    widgets.HTML("<h1>🌌 Quantum Echo Pattern Hunter</h1>"),
    widgets.HTML("<p><b>✅ FIXED for Colab</b> | Predict patterns using <u>echo resonance</u> feedback!</p>"),
    level_btns,
    widgets.HTML("<b>📈 Echo Visualizer</b>"),
    echo_visual,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>🔮 Pattern Scanner</b>"),
    input_box,
    progress_bar,
    status_out,
    widgets.HTML("<hr>"),
    widgets.HBox([widgets.VBox([widgets.HTML("<b>History</b>"), history_out]), 
                  widgets.VBox([widgets.HTML("<b>Score</b>"), score_out])]),
    restart_btn
], layout=widgets.Layout(width='950px', padding='10px'))

# Launch the FIXED game
display(game_ui)
start_game()
